ETAPE 1

###  Vérification locale avec pandas

Maintenant que le pipeline a tourné, utilisez les fonctionnalités natives de `dlt` pour interroger directement votre dataset BigQuery et récupérer les données de la table `bronze_products` sous forme de **DataFrame pandas**, afin de vérifier visuellement le résultat de votre ingestion depuis ce notebook.

In [ ]:
# Écrivez ici le code dlt pour charger la table bronze dans un DataFrame
import dlt

# Récupération du pipeline
pipeline = dlt.attach(pipeline_name="exam_pipeline")
# Lecture de la table bronze_products
pipeline.dataset().bronze_products.df()

/Users/lucasporcheron/Documents/Cours/OGEC/M2/BigData/cours_lasalle/big_data/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


,product_id,name,category,cost,retail_price,is_active,added_date,_dlt_load_id,_dlt_id
0,13010,Urban Sleep & Lounge 3010,NaN,19.8,46.10,True,2023-01-01,1791547399.804078,b3jIxpSG1AZ/Xw
1,16432,Casual Underwear 6432,NaN,37.2,78.32,False,2023-01-01,1791547399.804078,2B/yTgvQHrmtXQ
2,22277,Classic Maternity 12277,NaN,10.22,23.00,True,2023-01-01,1791547399.804078,qcFqmTnqJnO0kg
3,10094,Sporty Suits & Sport Coats 94,NaN,22.86,62.00,True,2023-01-02,1791547399.804078,4DxinhUJp2fUkw
4,10492,Urban Leggings 492,NaN,49.2,91.00,False,2023-01-02,1791547399.804078,lDFjoJN/NRb3kA
...,...,...,...,...,...,...,...,...,...
29995,15325,Vintage Socks & Hosiery 5325,underwear,30.22,57.90,False,2023-12-19,1791547399.804078,K/yzUpQh7K5r9Q
29996,18712,Casual Sleep & Lounge 8712,underwear,38.18,102.00,False,2023-12-22,1791547399.804078,Co59Vx3PbB1D/g
29997,22689,Urban Pants 12689,underwear,46.61,120.00,True,2023-12-24,1791547399.804078,9xpgMAzPaBGSvA
29998,32935,Casual Dresses 22935,underwear,33.74,83.00,False,2023-12-24,1791547399.804078,qOjdDpMKNckwug


In [8]:
# Récupérez ici des informations pertinentes sur l'execution du pipeline dlt
!dlt pipeline exam_pipeline info

Found pipeline exam_pipeline in /Users/lucasporcheron/.dlt/pipelines
Synchronized state:
_state_version: 1
_state_engine_version: 4
dataset_name: examen_lucas
default_schema_name: exam
schema_names: ['exam']
pipeline_name: exam_pipeline
destination_type: dlt.destinations.bigquery
destination_name: None
_version_hash: 4Og9G5/QGHL5Fxp4SW358XBJhkM6JEttmUK/Dmj+4Fc=

sources:
Add -v option to see sources state. Note that it could be large.

Local state:
first_run: False
_dev_mode: False
initial_cwd: /Users/lucasporcheron/Documents/Cours/OGEC/M2/BigData/cours_lasalle
last_run_context['uri']: file:///Users/lucasporcheron/Documents/Cours/OGEC/M2/BigData/cours_lasalle
_last_extracted_at: 2026-10-09 12:02:52.061796+00:00
_last_extracted_hash: 4Og9G5/QGHL5Fxp4SW358XBJhkM6JEttmUK/Dmj+4Fc=

Resources in schema: exam
read_json with 1 table(s) and 0 resource state slot(s)

Working dir content:
Has 2 completed load packages with following load ids:
1791547369.20804
1791547399.804078

Pipeline has last

-- Fichier: transformations/silver.sql
-- Nettoyage de bronze_products -> silver_products
CREATE OR REPLACE TABLE `lasalle-big-data.examen_lucas.silver_products` AS
SELECT
  SAFE_CAST(product_id AS INT64)                 AS product_id,  -- changement de type :texte -> entier
  name,                                                          -- déjà propre
  INITCAP(category)                        AS category,    -- uniformise la casse
  SAFE_CAST(NULLIF(cost, 'NULL') AS FLOAT64)     AS cout,        -- remplace le texte 'NULL' par un vrai NULL, puis texte -> nombre
  retail_price,                                                  -- déjà en FLOAT
  is_active,                                                     -- déjà en BOOLEAN
  SAFE_CAST(added_date AS DATE)                  AS added_date   -- changement de type texte -> date
FROM `lasalle-big-data.examen_lucas.bronze_products`;


-- Fichier: transformations/gold.sql
-- Agrégation par catégorie : silver_products -> gold_category_metrics
CREATE OR REPLACE TABLE `lasalle-big-data.examen_lucas.gold_category_metrics` AS
SELECT
  category,
  COUNT(*) AS total_products, -- nombre total de produits
  ROUND(AVG(retail_price), 2) AS prix_moyen, -- prix de vente moyen, arrondi à 2 décimales
  ROUND(AVG(retail_price - cout), 2) AS marge_moyenne -- marge unitaire moyenne,  arrondi à 2 décimales
FROM `lasalle-big-data.examen_lucas.silver_products`
GROUP BY category
ORDER BY marge_moyenne DESC; -- catégories les plus rentables en premier



# Fichier: .github/workflows/elt_pipeline.yml
name: Pipeline ELT Quotidien

# Le pipeline se lance manuellement, ou tous les jours à minuit
on:
  workflow_dispatch: # lancement manuel depuis l'onglet Actions
  schedule:
    - cron: "0 0 * * *" # Run  Tous les jours à minuit

jobs:
  run-elt:
    runs-on: ubuntu-latest # machine virtuelle Linux fournie par GitHub
    
    # Permissions requises pour utiliser Workload Identity Federation (OIDC) avec GitHub Actions
    permissions:
      contents: 'read'
      id-token: 'write'

    steps:
    # Récupère le code du dépôt sur la machine
      - name: Récupérer le code (Checkout)
        uses: actions/checkout@v4

    # Connexion à Google Cloud
      - name: Authentification Google Cloud (WIF)
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: 'projects/833312929773/locations/global/workloadIdentityPools/github-pool/providers/github-provider'
          service_account: 'github-actions-elt@lasalle-big-data.iam.gserviceaccount.com'

    # Installe Python puis la librairie dlt 
      - name: Installation de Python et dlt
        uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: |
          uv add "dlt[bigquery,gs]"         

      #--- 🟢 ÉTAPE BRONZE : INGESTION ---
    # Charge le JSON du bucket dans la table bronze_products
      - name: 1. Ingestion API vers Bronze
        run: python ingestion/load_products.py

      # --- ⚪ ÉTAPE SILVER : TRANSFORMATION ---
    # Nettoie les données : bronze_products -> silver_products
      - name: 2. ransformation Bronze vers Silver
        run: bq query --use_legacy_sql=false < transformations/silver.sql


      # --- 🟡 ÉTAPE GOLD : AGRÉGATION ---
    # Calcule les indicateurs par catégorie : silver_products -> gold_category_metrics
      - name: 3. Agrégation Silver vers Gold
        run: bq query --use_legacy_sql=false < transformations/gold.sql

